# Camada Silver: Projeto CineData Analytics

**Objetivo:** limpar, tipar, deduplicar e traduzir os dados da Bronze (que **não é alterada**), gerando 7 tabelas Silver com nomes de colunas em **português** e **tipagem correta**.

**Princípios aplicados**

* **Conversão segura de tipos:** o workspace Databricks costuma rodar com `ANSI mode` ligado, em que `CAST` inválido derruba o pipeline. Por isso toda conversão numérica é protegida por regex (somente converte se o texto tem formato válido, caso contrário vira `NULL`), e datas usam `try_to_timestamp`.
* **Valores de ausência** (`Unknown`, `Não Informado`, `N/A`, `Nenhum`, `[]`, vazio…) são tratados como `NULL`.
* **Deduplicação** sempre mantém a versão com **maior `ingestion_datetime`**, em empate usa-se um critério determinístico.
* **Ordem de execução**: a tabela de cotação é construída **antes** da financeira, pois a conversão para BRL depende dela.

In [0]:
from decimal import Decimal
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalogo", "", "")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if CATALOGO:
    spark.sql(f"USE CATALOG {CATALOGO}")

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Evita erro de parse de datas ambíguas entre o parser novo e o legado.
try:
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
except Exception:
    pass # algumas configurações não são editáveis em compute serverless

# Textos que representam 'dado ausente' na origem
NULOS_TEXTUAIS = ["unknown", "não informado", "nao informado", "n/a", "null", "none",
                  "nan", "nenhum", "", "-", "--", "[]"]


def nulificar_ausentes(c):
    # trim + textos de ausência (Unknown, N/A, Não Informado...) vira NULL
    s = F.trim(c)
    return F.when(F.lower(s).isin(NULOS_TEXTUAIS), F.lit(None).cast("string")).otherwise(s)


def para_double(c):
    # Texto vai pra DOUBLE de forma segura. Aceita '9.9', '9,9' (vírgula decimal) e '1.234,56'.
    # Qualquer texto fora do formato numérico vira NULL.
    s = F.trim(c)
    s = (F.when(s.rlike(r"^-?[0-9]{1,3}(\.[0-9]{3})+,[0-9]+$"),
               F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", ".")) # 1.234,56 -> 1234.56
          .when(s.rlike(r"^-?[0-9]+,[0-9]+$"), F.regexp_replace(s, ",", ".")) # 9,9 -> 9.9
          .otherwise(s))
    return F.when(s.rlike(r"^-?[0-9]{1,15}(\.[0-9]+)?$"), s.cast("double"))


def para_int(c):
    # Texto vai pra INT de forma segura. Aceita '123' e '123.0'; rejeita '8.4', textos e overflow.
    s = F.trim(c)
    return F.when(s.rlike(r"^-?[0-9]{1,9}(\.0+)?$"),
                  F.regexp_replace(s, r"\.0+$", "").cast("int"))


def deduplicar(df, chave, desempate=()):
    # Mantém 1 linha por chave: maior ingestion_datetime, empate -> critérios de 'desempate'.
    ordem = [F.col("ingestion_datetime").desc()] + list(desempate)
    w = Window.partitionBy(*chave).orderBy(*ordem)
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


def completude(colunas):
    # Nº de colunas preenchidas (usado para preferir a versão mais completa em empates).
    return sum((F.col(c).isNotNull().cast("int") for c in colunas), F.lit(0))


def gravar_silver(df, tabela):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(f"silver.{tabela}"))
    print(f"silver.{tabela}: {spark.table('silver.' + tabela).count():,} linhas")


ID_VALIDO = r"^[0-9]+$" # id de filme: somente dígitos (descarta ids corrompidos por Column Shift)

## `silver.tb_cotacao_dolar`: série temporal contínua com Forward Fill

Regras:
1. A PTAX devolve **vários boletins por dia**, o valor oficial do dia é o **último boletim** (maior `dataHoraCotacao`).
2. Constrói-se um **calendário contínuo** (um registro por dia corrido) e aplica-se **Forward Fill**: finais de semana e feriados recebem a cotação do último dia útil anterior.
3. O calendário vai da 1ª cotação até a data da última ingestão, cobrindo também finais de semana "à direita" (exemplo: carga rodada num sábado).

In [0]:
b_cot = spark.table("bronze.tb_cotacao_dolar")

cot = (
    b_cot
    .select(
        F.expr("try_cast(trim(dataHoraCotacao) as timestamp)").alias("data_hora_cotacao"),
        para_double(F.col("cotacaoCompra")).alias("cotacao_dolar"),
        F.col("ingestion_datetime"),
    )
    .filter(F.col("data_hora_cotacao").isNotNull() & F.col("cotacao_dolar").isNotNull() & (F.col("cotacao_dolar") > 0))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
)

# 1 cotação por dia: último boletim do dia (empate -> ingestão mais recente)
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(), F.col("ingestion_datetime").desc())
cot_dia = (cot.withColumn("_rn", F.row_number().over(w_dia)).filter("_rn = 1")
              .select("data_cotacao", "cotacao_dolar"))

lim = (cot.agg(F.min("data_cotacao").alias("ini"),
               F.max("data_cotacao").alias("fim_cotacao"),
               F.max(F.to_date("ingestion_datetime")).alias("fim_ingestao")).first())
if lim["ini"] is None:
    raise ValueError("bronze.tb_cotacao_dolar não possui cotações válidas.")
data_fim_calendario = max(lim["fim_cotacao"], lim["fim_ingestao"])

# Calendário contínuo (1 linha por dia corrido)
calendario = spark.sql(
    f"SELECT explode(sequence(to_date('{lim['ini']}'), to_date('{data_fim_calendario}'), interval 1 day)) AS data_cotacao"
)

# Forward Fill: último valor não-nulo até a linha corrente
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
cot_dia = cot_dia.withColumn("_data_origem", F.col("data_cotacao"))

silver_cotacao = (
    calendario.join(cot_dia, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_dolar").isNull()) # True = veio do Forward Fill
    .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ff))
    .withColumn("data_referencia_cotacao", F.last("_data_origem", ignorenulls=True).over(w_ff))
    .select(
        "data_cotacao",
        F.col("cotacao_dolar").cast("decimal(12,4)").alias("cotacao_dolar"),
        "cotacao_preenchida",
        "data_referencia_cotacao",
    )
    .orderBy("data_cotacao")
)

gravar_silver(silver_cotacao, "tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver.tb_cotacao_dolar: 13 linhas


data_cotacao,cotacao_dolar,cotacao_preenchida,data_referencia_cotacao
2026-09-25,5.1985,false,2026-09-25
2026-09-26,5.1985,true,2026-09-25
2026-09-27,5.1985,true,2026-09-25
2026-09-28,5.2126,false,2026-09-28
2026-09-29,5.2198,false,2026-09-29
2026-09-30,5.1803,false,2026-09-30
2026-10-01,5.2073,false,2026-10-01
2026-10-02,5.2232,false,2026-10-02
2026-10-03,5.2232,true,2026-10-02
2026-10-04,5.2232,true,2026-10-02


## `silver.tb_info_filmes` (vem de: `bronze.tb_movies_info`)

| Regra | Implementação |
|---|---|
| **Status** | Normaliza (`lower`, `trim`, hífens/underscores → espaço, espaços duplos colapsados) → traduz (`released`→Lançado, `post production`→Pós-Produção, `in production`→Em Produção, `planned`→Planejado, `rumored`→Rumores, `canceled`→Cancelado). Qualquer outro valor (inclusive linhas corrompidas, cujo `status` virou data) → **"Não Informado"**. |
| **Data multi-formato** | Padrões encontrados na origem: `yyyy-MM-dd`, `dd/MM/yyyy` e `MM-dd-yyyy`. O separador (`/` vs `-`) define o formato; confirmado nos dados: com `/` nunca há valor >12 na 2ª posição (é dia/mês), com `-` nunca há valor >12 na 1ª posição (é mês-dia). Há *fallbacks* para os formatos inversos, só vira `NULL` se **nenhum** padrão converter (exemplo: texto de sinopse deslocado, 31/02). |
| **Deduplicação** | 1 registro por `id_filme`, mantendo o de maior `ingestion_datetime`. Em empate: versão mais completa e, por fim, a data mais recente (determinístico). As duplicatas da origem diferem só no formato de data/caixa do status, então a escolha não altera o conteúdo. |
| **Tipagem** | `duracao_minutos` INT (texto inválido → NULL), `data_lancamento` DATE, `ano_lancamento` INT derivado. Textos vazios → NULL. |

In [0]:
b_info = spark.table("bronze.tb_movies_info")

# Normalização + tradução do status
status_norm = F.lower(F.trim(F.col("status")))
status_norm = F.regexp_replace(status_norm, r"[-_]+", " ") # 'in-production' -> 'in production'
status_norm = F.trim(F.regexp_replace(status_norm, r"\s+", " ")) # colapsa espaços

MAPA_STATUS = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado", # grafia extra
}
status_traduzido = None
for k, v in MAPA_STATUS.items():
    status_traduzido = F.when(status_norm == k, v) if status_traduzido is None else status_traduzido.when(status_norm == k, v)
status_traduzido = status_traduzido.otherwise("Não Informado") # corrompido / não mapeável


# Conversão de data multi-formato
def converter_data(df, origem, destino):
    # Testa os padrões em ordem de prioridade, o 1º que converter vence. Sem match -> NULL.
    s = f"trim(`{origem}`)"
    padroes = [
        ("^[0-9]{4}-[0-9]{2}-[0-9]{2}$", "yyyy-MM-dd"),
        ("^[0-9]{2}/[0-9]{2}/[0-9]{4}$", "dd/MM/yyyy"),
        ("^[0-9]{2}-[0-9]{2}-[0-9]{4}$", "MM-dd-yyyy"),
        ("^[0-9]{2}/[0-9]{2}/[0-9]{4}$", "MM/dd/yyyy"), # fallback: 02/13/2020
        ("^[0-9]{2}-[0-9]{2}-[0-9]{4}$", "dd-MM-yyyy"), # fallback: 25-04-2018
    ]
    tentativas = [
        f"CASE WHEN {s} RLIKE '{rx}' THEN CAST(try_to_timestamp({s}, '{fmt}') AS DATE) END"
        for rx, fmt in padroes
    ]
    return df.withColumn(destino, F.expr("coalesce(" + ", ".join(tentativas) + ")"))


def texto_ou_nulo(c):
    s = F.trim(c)
    return F.when(s == "", F.lit(None).cast("string")).otherwise(s)


info = (
    b_info
    .filter(F.col("id").rlike(r"^\s*[0-9]+\s*$"))
    .select(
        F.trim("id").alias("id_filme"),
        texto_ou_nulo(F.col("title")).alias("titulo"),
        texto_ou_nulo(F.col("original_title")).alias("titulo_original"),
        F.col("release_date").alias("_data_bruta"),
        para_int(F.col("runtime")).alias("duracao_minutos"),
        # código de idioma ISO (2-3 letras), textos deslocados viram NULL
        F.when(F.lower(F.trim("original_language")).rlike(r"^[a-z]{2,3}$"), F.lower(F.trim("original_language"))).alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        texto_ou_nulo(F.col("overview")).alias("sinopse"),
        texto_ou_nulo(F.col("tagline")).alias("frase_divulgacao"),
        "ingestion_datetime",
    )
)
info = converter_data(info, "_data_bruta", "data_lancamento").drop("_data_bruta")

# Deduplicação: versão mais recente por filme
colunas_info = ["titulo", "titulo_original", "data_lancamento", "duracao_minutos",
                "idioma_original", "status_filme", "sinopse", "frase_divulgacao"]
info = (
    info.withColumn("_completude", completude(colunas_info))
    .transform(lambda d: deduplicar(d, ["id_filme"],
               desempate=[F.col("_completude").desc(), F.col("data_lancamento").desc_nulls_last(), F.col("titulo").asc_nulls_last()]))
    .drop("_completude")
)

silver_info = info.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "duracao_minutos",
    "idioma_original", "status_filme", "sinopse", "frase_divulgacao",
    F.year("data_lancamento").alias("ano_lancamento"),
)
gravar_silver(silver_info, "tb_info_filmes")

display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))
display(spark.table("silver.tb_info_filmes").limit(10))

silver.tb_info_filmes: 97,611 linhas


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000014,On va manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,0,es,Lançado,null,null,2021
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,2016
1000073,A Chance To Win,Pour l'honneur,2023-05-03,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,2023
1000091,Amy Miller: Ham Mouth,Amy Miller: Ham Mouth,2022-03-24,35,en,Lançado,"Amy Miller reflects on her recent breakup, shares her love of baths and reveals the 40-year-old shit she does.",null,2022
1000092,Nowhere,Invelle,2023-09-08,92,it,Lançado,"Three different epochs through the gaze of three children: Zelinda, who loses her mother to the Spanish Flu during World War I and sees the specter of Nazism loom; Assunta, who lives during the Nazi occupation between bombings, raids, and executions; Icaro, who abandons the countryside during the Years of Lead (“Anni di Piombo”) and accepts a new life. A story across the difficulties and the troubles of the 20th century between memories, affection, nostalgia, and gratitude.",null,2023
1000096,Critical Keertanegalu,Critical Keertanegalu,2022-07-15,129,kn,Lançado,"A Lawyer called Kariyappa, he has been instrumental in filing the flaws in society, During IPL season, Gambling became a reality and millions became victims of the year. Realising that he comes to court to file an appeal against IPL for the of ban the IPL. He describes the four gambling events happened in Karnataka. All this as lawyer Kariyappa puts it infront of judge.",null,2022
1000099,Chowdhury Rajbari,Chowdhury Rajbari,2017-12-08,131,bn,Lançado,"Giaa and Arnab are a married couple, who visit Arnab's ancestral palace 'Chowdhury Rajbari' for a romantic getaway. Once there, Giaa experiences strange occurrences and memories of a past life. In her search for truth, Giaa comes across Dr. Oindrila, who assists her in her search. But trouble starts when they come across more questions, instead of answers.",null,2017
1000108,Le Petit blond de la casbah,Le Petit blond de la casbah,2023-11-15,0,fr,Pós-Produção,null,null,2023


## `silver.tb_financeiro_filmes` (vem de: `bronze.tb_movies_financials`)

| Regra | Implementação |
|---|---|
| **Ausência** | `Unknown`, `Não Informado`, `N/A` → `NULL` **antes** da conversão de tipo. |
| **Higienização** | Remove `USD`, `$`, `R$`, espaços e separadores de milhar. Formato abreviado `34.0M` (visto na origem) é expandido para 34.000.000, apenas remover letras geraria 34,0 (valor errado por 6 ordens de grandeza). |
| **Tipo** | `DECIMAL(18,2)`; valores **≤ 0** (zeros e negativos, exemplo: receita `-800526015`) → `NULL`. |
| **BRL** | `valor_usd × cotação` usando a **cotação mais recente** da `silver.tb_cotacao_dolar` (a API traz só 7 dias, então não há cotação histórica por data de lançamento, usa-se uma taxa única e explícita). |
| **Lucro / Margem** | `lucro = receita − orçamento` **somente se os dois existem** (se um deles é `NULL` o lucro é `NULL`; tratar o ausente como 0 inventaria um prejuízo/lucro falso). `margem_lucro_percentual = lucro / receita × 100`, com `try_divide` (retorna `NULL` em divisão por zero, sem erro). |
| **Deduplicação** | A origem repete filmes (exemplo: `1.3M` e `1250000` para o mesmo id). Mantém 1 linha por `id_filme` para garantir o **grão único** exigido na tabela fato: maior `ingestion_datetime`, empate → versão com mais campos preenchidos, depois maior orçamento/receita. |

In [0]:
b_fin = spark.table("bronze.tb_movies_financials")

def limpar_moeda(c):
    # Texto monetário sujo -> DECIMAL(18,2) (ou NULL). Valores <= 0 viram NULL.
    s = F.upper(nulificar_ausentes(c)) # 'Unknown'/'N/A'/... -> NULL
    s = F.regexp_replace(s, r"(USD|US\$|R\$|BRL|EUR|\$|€)", "") # símbolos/códigos de moeda
    s = F.regexp_replace(s, r"\s+", "") # espaços
    eh_milhao = s.rlike(r"^-?[0-9]+([.,][0-9]+)?M$") # '34.0M' = 34 milhões
    s = F.regexp_replace(s, r"M$", "")
    s = F.regexp_replace(s, r"(?<=[0-9])[.,](?=[0-9]{3}(?![0-9]))", "") # separador de milhar
    s = F.regexp_replace(s, ",", ".") # vírgula decimal restante
    base = F.when(s.rlike(r"^-?[0-9]{1,15}(\.[0-9]+)?$"), s.cast("decimal(18,2)"))
    valor = F.when(eh_milhao, base * F.lit(1000000).cast("decimal(18,2)")).otherwise(base).cast("decimal(18,2)")
    return F.when(valor > 0, valor) # zero/negativo -> NULL


# Cotação a aplicar: a mais recente da série contínua (Silver)
ultima_cotacao = spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()
TAXA_USD_BRL = ultima_cotacao["cotacao_dolar"] # Decimal
print(f"Cotação aplicada: R$ {TAXA_USD_BRL} (referência {ultima_cotacao['data_referencia_cotacao']})")

fin = (
    b_fin
    .filter(F.col("id").rlike(r"^\s*[0-9]+\s*$"))
    .select(
        F.trim("id").alias("id_filme"),
        limpar_moeda(F.col("budget")).alias("orcamento_usd"),
        limpar_moeda(F.col("revenue")).alias("receita_usd"),
        "ingestion_datetime",
    )
    .withColumn("_completude", completude(["orcamento_usd", "receita_usd"]))
)
fin = deduplicar(fin, ["id_filme"], desempate=[
    F.col("_completude").desc(), F.col("orcamento_usd").desc_nulls_last(), F.col("receita_usd").desc_nulls_last()
]).drop("_completude")

taxa = F.lit(TAXA_USD_BRL)
silver_fin = (
    fin
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * taxa, 2).cast("decimal(18,2)"))
    .withColumn("receita_brl",   F.round(F.col("receita_usd") * taxa, 2).cast("decimal(18,2)"))
    # Lucro só existe quando receita e orçamento existem (sem inventar zeros)
    .withColumn("lucro_usd", F.when(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
                                    F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", F.when(F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
                                    F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem % = lucro / receita * 100  (try_divide: divisão por zero -> NULL, sem erro)
    .withColumn("margem_lucro_percentual",
                F.round(F.expr("try_divide(lucro_usd, receita_usd)") * 100, 2).cast("double"))
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_percentual")
)
gravar_silver(silver_fin, "tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(10))

Cotação aplicada: R$ 4.9929 (referência 2026-10-07)
silver.tb_financeiro_filmes: 99,006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000054,null,null,null,null,null,null,null
1000058,4700000.00,null,23466630.00,null,null,null,null
1000059,null,null,null,null,null,null,null
1000073,6000000.00,null,29957400.00,null,null,null,null


## `silver.tb_metricas_engajamento` (origem: `bronze.tb_movies_metrics`)

| Regra | Implementação |
|---|---|
| **Popularidade** | Aceita ponto **e** vírgula decimal (`"154,34"` → 154.34) antes do cast, para não virar `NULL` silenciosamente. |
| **Column Shift** | Linhas em que trechos de sinopse/nomes caíram nas colunas numéricas: conversão **segura** (regex + cast) → texto vira `NULL`, sem quebrar o pipeline. `qtd_votos_*` só aceita inteiro (`8.4` numa coluna de contagem é valor deslocado → `NULL`). |
| **Linha deslocada** | Se `averageRating`/`numVotes` contêm texto, a linha está desalinhada; nela, uma popularidade inteira sem decimais (ex.: `2020`, `1969` = ano deslocado) é invalidada (`NULL`). Afeta 3 linhas e evita que apareçam no ranking de popularidade. |
| **Limites de negócio** | Notas TMDB/IMDb fora de **[0, 10]** (inclui erro de escala ×10, ex.: `79.97`, `85.0`, `100.0`) → `NULL`. Popularidade ou contagem de votos **negativas** → `NULL`. |
| **Deduplicação** | A origem repete filmes (ex.: `vote_average` 8.5 e 85.0 para o mesmo id). A limpeza é feita **antes** da deduplicação, então a versão inválida perde para a completa. Critério: maior `ingestion_datetime` → mais campos válidos. |

In [0]:
b_met = spark.table("bronze.tb_movies_metrics")


def nota_valida(c):
    v = para_double(c)
    return F.when((v >= 0) & (v <= 10), v) # escala 0-10; fora disso (ex.: 79.97) -> NULL


def nao_negativo_double(c):
    v = para_double(c)
    return F.when(v >= 0, v)


def nao_negativo_int(c):
    v = para_int(c)
    return F.when(v >= 0, v)


# Detecção de Column Shift: se uma coluna que deveria ser numérica (averageRating / numVotes) contém TEXTO,
# a linha está desalinhada. Nessas linhas, uma popularidade sem parte decimal (ex.: '2020', '1969' - um ano
# deslocado, padrão inexistente numa popularidade TMDB real) é descartada, senão poluiria rankings de popularidade.
def _texto_nao_numerico(c):
    t = F.trim(c)
    return (t != "") & ~t.rlike(r"^-?[0-9]+([.,][0-9]+)?$")

linha_deslocada = _texto_nao_numerico(F.col("averageRating")) | _texto_nao_numerico(F.col("numVotes"))
popularidade_ok = F.when(linha_deslocada & F.trim(F.col("popularity")).rlike(r"^[0-9]+$"), F.lit(None).cast("double")) \
                   .otherwise(nao_negativo_double(F.col("popularity")))

met = (
    b_met
    .filter(F.col("id").rlike(r"^\s*[0-9]+\s*$"))
    .select(
        F.trim("id").alias("id_filme"),
        popularidade_ok.alias("popularidade"),
        nota_valida(F.col("vote_average")).alias("nota_media_tmdb"),
        nao_negativo_int(F.col("vote_count")).alias("qtd_votos_tmdb"),
        nota_valida(F.col("averageRating")).alias("nota_media_imdb"),
        nao_negativo_int(F.col("numVotes")).alias("qtd_votos_imdb"),
        "ingestion_datetime",
    )
    .withColumn("_completude", completude(["popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
                                           "nota_media_imdb", "qtd_votos_imdb"]))
)
silver_met = (
    deduplicar(met, ["id_filme"], desempate=[F.col("_completude").desc(),
               F.col("qtd_votos_imdb").desc_nulls_last(), F.col("qtd_votos_tmdb").desc_nulls_last(),
               F.col("popularidade").desc_nulls_last()])
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)
gravar_silver(silver_met, "tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").orderBy(F.desc("popularidade")).limit(10))

silver.tb_metricas_engajamento: 95,115 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
565770,2994.357,7.139,1023,5.9,119501
980489,2680.593,8.068,null,7.1,138696
968051,1692.778,6.545,365,5.6,84018
615656,1567.273,6.912,2034,null,105031
762430,1547.22,6.816,258,null,29152
1008042,1458.514,7.214,973,7.1,224148
385687,1175.267,null,3881,5.7,149785
678512,1111.036,7.973,503,7.6,null
346698,1069.34,7.279,5074,6.8,704472
976573,1008.942,7.757,2467,7.0,169047


## `silver.tb_avaliacoes_usuarios` (origem: `bronze.tb_movies_reviews`)

* **Nota** convertida com cast seguro (aceita vírgula decimal) e restrita à escala **0–10**; fora da faixa → `NULL`.
* **Comentário** nulo, vazio ou só com espaços → `"Sem comentário"`.
* **Duplicatas integrais** (filme + usuário + nota + comentário idênticos) removidas **após** a limpeza, para que `''` e `'   '` sejam reconhecidos como o mesmo comentário.

In [0]:
b_rev = spark.table("bronze.tb_movies_reviews")

nota_u = para_double(F.col("nota"))
rev = (
    b_rev
    .filter(F.col("id").rlike(r"^\s*[0-9]+\s*$"))
    .select(
        F.trim("id").alias("id_filme"),
        texto_ou_nulo(F.col("nome")).alias("nome_usuario"),
        F.when((nota_u >= 0) & (nota_u <= 10), nota_u).alias("nota_usuario"),
        F.coalesce(texto_ou_nulo(F.col("comentario")), F.lit("Sem comentário")).alias("comentario_usuario"),
    )
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)
gravar_silver(rev, "tb_avaliacoes_usuarios")
display(spark.table("silver.tb_avaliacoes_usuarios").limit(10))

silver.tb_avaliacoes_usuarios: 32,412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


## `silver.tb_generos` (origem: `bronze.tb_credits_and_tags`, coluna `genres`)

1. **Separadores**: `;` e `|` são convertidos para `,` (a origem mistura vírgula, ponto e vírgula e também `|`) e então `split` + `explode` → 1 linha por filme/gênero.
2. **Column Shift**: na coluna `genres` aparecem valores deslocados (`0.6`, caminhos `/abc.jpg`, trechos de sinopse/tagline). Só são mantidos valores que pertencem ao **domínio de gêneros** (os 19 gêneros que aparecem na base, padrão TMDB), comparados sem diferenciar maiúsculas/minúsculas e gravados padronizados.
3. Duplicatas (filme × gênero) removidas.

In [0]:
b_cred = spark.table("bronze.tb_credits_and_tags").filter(F.col("id").rlike(r"^\s*[0-9]+\s*$"))

GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family",
    "Fantasy", "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western",
]
df_dominio = spark.createDataFrame([(g.lower(), g) for g in GENEROS_VALIDOS], ["_chave", "genero"])

separadores_normalizados = F.regexp_replace(F.col("genres"), r"[;|]", ",")     # ';' e '|' -> ','
generos = (
    b_cred
    .select(F.trim("id").alias("id_filme"),
            F.explode(F.split(separadores_normalizados, ",")).alias("_bruto"))
    .withColumn("_chave", F.lower(F.trim("_bruto")))
    .join(F.broadcast(df_dominio), "_chave", "inner") # descarta resíduos / valores deslocados
    .select("id_filme", "genero")
    .dropDuplicates()
)
gravar_silver(generos, "tb_generos")
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

silver.tb_generos: 141,941 linhas


genero,count
Drama,32615
Documentary,19235
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
TV Movie,4115


## `silver.tb_pessoas_empresas` (origem: `bronze.tb_credits_and_tags`)

Consolida em uma única tabela: `cast`→**Ator**, `directors`→**Diretor**, `writers`→**Roteirista**, `production_companies`→**Produtora**.

Limpeza (as colunas têm *Column Shift*: números, trechos de sinopse, palavras-chave, idiomas, gêneros e caminhos de imagem caíram nelas):

* Remove aspas/barras nas pontas, `NULL` para `[]`, `N/A`, `Nenhum`, vazio.
* Descarta tokens **numéricos**, caminhos (`/xyz.jpg`), com aspas/barras/quebra de linha internas (`Drama\n428703` = linhas engolidas por aspas quebradas), sufixos órfãos do split (`Jr`, `Sr`), e **fragmentos de frase** (começam em minúscula e têm ≥ 4 palavras, ou têm ≥ 13 palavras).
* Descarta tokens que pertencem ao **vocabulário de outras colunas da própria origem** (idiomas, países, gêneros e — para tokens 100% minúsculos — palavras-chave), pois são valores deslocados (ex.: diretor `English`, produtora `Documentary`).
* Padroniza capitalização com `initcap` e elimina duplicatas (`id_filme` × nome × tipo).

> Limitação conhecida: `initcap` transforma `McDonald`→`Mcdonald` e `ABBA`→`Abba`.

In [0]:
def vocabulario(df, coluna, minimo):
    # Tokens (minúsculos) de uma coluna da origem que aparecem >= 'minimo' vezes.
    sep = F.regexp_replace(F.col(coluna), r"[;|]", ",")
    return (df.select(F.explode(F.split(sep, ",")).alias("v"))
              .select(F.lower(F.trim("v")).alias("v"))
              .groupBy("v").count().filter(F.col("count") >= minimo).select("v"))


vocab_dominio = (
    vocabulario(b_cred, "spoken_languages", 50)
    .union(vocabulario(b_cred, "production_countries", 50))
    .union(spark.createDataFrame([(g.lower(),) for g in GENEROS_VALIDOS], ["v"]))
    .distinct().withColumn("_in_dominio", F.lit(1)).withColumnRenamed("v", "_chave")
)
vocab_keywords = vocabulario(b_cred, "keywords", 20).withColumn("_in_kw", F.lit(1)).withColumnRenamed("v", "_chave")

ENTIDADES = [("cast", "Ator"), ("directors", "Diretor"), ("writers", "Roteirista"), ("production_companies", "Produtora")]


def explodir_entidade(coluna, tipo):
    sep = F.regexp_replace(F.col(coluna), r"[;|]", ",")
    return (b_cred.select(F.trim("id").alias("id_filme"), F.explode(F.split(sep, ",")).alias("_bruto"))
                  .withColumn("tipo_entidade", F.lit(tipo)))


bruto = None
for coluna, tipo in ENTIDADES:
    d = explodir_entidade(coluna, tipo)
    bruto = d if bruto is None else bruto.unionByName(d)

PONTAS = "^[\\s\"'\\\\]+|[\\s\"'\\\\]+$" # espaços/aspas/barras nas pontas
nome = nulificar_ausentes(F.regexp_replace(F.trim("_bruto"), PONTAS, ""))
n_palavras = F.size(F.split(F.trim(nome), r"\s+"))

candidatos = (
    bruto.withColumn("_nome", nome)
    .withColumn("_n_palavras", n_palavras)
    .filter(F.col("_nome").isNotNull())
    .filter(~F.col("_nome").rlike(r"^[0-9.,\s-]+$")) # só números (valores deslocados)
    .filter(~F.col("_nome").rlike(r"[\\\"]")) # restos de aspas/barras internas
    .filter(~F.col("_nome").rlike(r"[\r\n]")) # 'Drama\n428703': aspas quebradas engoliram linhas
    .filter(~F.lower(F.col("_nome")).isin("jr", "jr.", "sr", "sr.", "dr", "dr.", "ii", "iii", "iv")) # sufixo órfão do split por vírgula (ex.: 'Davis, Jr.')
    .filter(~F.col("_nome").rlike(r"(^/)|(\.(jpg|jpeg|png)$)")) # caminhos de imagem
    .filter(F.col("_n_palavras") < 13) # frases longas = sinopse deslocada
    .filter(~(F.col("_nome").rlike(r"^[a-z]") & (F.col("_n_palavras") >= 4))) # fragmento de frase
    .filter(~(F.col("_nome").rlike(r"\.$") & (F.col("_n_palavras") >= 5))) # frase terminada em ponto
    .withColumn("_chave", F.lower(F.col("_nome")))
    .withColumn("_so_minusculo", F.col("_nome") == F.lower(F.col("_nome")))
)

limpo = (
    candidatos
    .join(F.broadcast(vocab_dominio), "_chave", "left")
    .join(vocab_keywords, "_chave", "left")
    .filter(F.col("_in_dominio").isNull()) # idioma/país/gênero deslocado
    .filter(~(F.col("_in_kw").isNotNull() & F.col("_so_minusculo"))) # palavra-chave deslocada (minúscula)
    .select("id_filme",
            F.initcap(F.col("_nome")).alias("nome_entidade"), # padroniza capitalização
            "tipo_entidade")
    .dropDuplicates()
)
gravar_silver(limpo, "tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").agg(
    F.count("*").alias("qtd_registros"), F.countDistinct("nome_entidade").alias("qtd_entidades_distintas")))
display(spark.table("silver.tb_pessoas_empresas").limit(10))

silver.tb_pessoas_empresas: 880,749 linhas


tipo_entidade,qtd_registros,qtd_entidades_distintas
Produtora,115023,44543
Roteirista,125080,84729
Diretor,94761,63604
Ator,545885,269060


id_filme,nome_entidade,tipo_entidade
447332,Rhoda Pell,Ator
427641,Dwayne Johnson,Ator
207932,Tom Hanks,Ator
447365,Karen Gillan,Ator
453405,Victor Hugo,Ator
479455,Liam Neeson,Ator
752623,Bowen Yang,Ator
393519,Marouan Iddoub,Ator
14564,Zach Roerig,Ator
137116,Mandy Patinkin,Ator


## Validação final da Silver (checks de qualidade)

In [0]:
def checar(nome_check, ok):
    print(("Foi" if ok else "Não foi"), nome_check)
    assert ok, f"Falhou: {nome_check}"


def unico(tabela, chave):
    t = spark.table(tabela)
    return t.count() == t.select(*chave).distinct().count()


checar("tb_info_filmes: 1 linha por id_filme",              unico("silver.tb_info_filmes", ["id_filme"]))
checar("tb_financeiro_filmes: 1 linha por id_filme",        unico("silver.tb_financeiro_filmes", ["id_filme"]))
checar("tb_metricas_engajamento: 1 linha por id_filme",     unico("silver.tb_metricas_engajamento", ["id_filme"]))
checar("tb_cotacao_dolar: 1 linha por dia e sem lacunas (nenhuma cotação nula)",
       unico("silver.tb_cotacao_dolar", ["data_cotacao"]) and
       spark.table("silver.tb_cotacao_dolar").filter("cotacao_dolar IS NULL").count() == 0)
checar("tb_generos: sem duplicatas filme x gênero",         unico("silver.tb_generos", ["id_filme", "genero"]))
checar("tb_pessoas_empresas: sem duplicatas",               unico("silver.tb_pessoas_empresas", ["id_filme", "nome_entidade", "tipo_entidade"]))
checar("tb_avaliacoes_usuarios: sem duplicatas integrais",  unico("silver.tb_avaliacoes_usuarios", ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]))
checar("status_filme só contém valores traduzidos",
       spark.table("silver.tb_info_filmes").filter(~F.col("status_filme").isin(
           "Lançado", "Pós-Produção", "Em Produção", "Planejado", "Rumores", "Cancelado", "Não Informado")).count() == 0)
checar("notas TMDB/IMDb dentro de [0,10]",
       spark.table("silver.tb_metricas_engajamento").filter(
           "nota_media_tmdb NOT BETWEEN 0 AND 10 OR nota_media_imdb NOT BETWEEN 0 AND 10").count() == 0)
checar("nota de usuário dentro de [0,10]",
       spark.table("silver.tb_avaliacoes_usuarios").filter("nota_usuario NOT BETWEEN 0 AND 10").count() == 0)
checar("orçamento/receita > 0 quando preenchidos",
       spark.table("silver.tb_financeiro_filmes").filter("orcamento_usd <= 0 OR receita_usd <= 0").count() == 0)

Foi tb_info_filmes: 1 linha por id_filme
Foi tb_financeiro_filmes: 1 linha por id_filme
Foi tb_metricas_engajamento: 1 linha por id_filme
Foi tb_cotacao_dolar: 1 linha por dia e sem lacunas (nenhuma cotação nula)
Foi tb_generos: sem duplicatas filme x gênero
Foi tb_pessoas_empresas: sem duplicatas
Foi tb_avaliacoes_usuarios: sem duplicatas integrais
Foi status_filme só contém valores traduzidos
Foi notas TMDB/IMDb dentro de [0,10]
Foi nota de usuário dentro de [0,10]
Foi orçamento/receita > 0 quando preenchidos
